In [ ]:
from pathlib import Path

import joblib

import numpy as np
import pandas as pd
from pyscipopt import Model, quicksum

ROOT = Path("../..")

preds = pd.read_csv(ROOT / "models/predictions.csv")

all_data = pd.read_csv(
    ROOT / "inputs/rolled_player_data_23-24_all_players_xp_fixed.csv"
)
preds["value"] = preds["value"] / 10
# preds = preds[preds["minutes"] >= 60]

# all_data = all_data.rename(columns={"position": "pos"})

preds = preds.merge(
    all_data[["fpl_id", "round", "minutes_rolling_3"]],
    on=["fpl_id", "round"],
    how="left",
)

preds = preds[preds["minutes_rolling_3"] > 45]  # .sample(50)
preds

,pos,round,fpl_id,value,name,team,total_points,selected_at,model,pred,xP,in_selection,minutes,played,minutes_rolling_3
0,GK,5,17,5.0,Aaron Ramsdale,Arsenal,0,5,hgb,4.753778,2.7,True,0.0,False,90.000000
2,GK,5,28,4.0,Matt Turner,Nott'm Forest,2,5,hgb,3.438714,3.8,True,90.0,True,90.000000
3,GK,5,49,4.9,Emiliano Martínez Romero,Aston Villa,2,5,hgb,3.199263,1.0,True,90.0,True,60.000000
6,GK,5,77,4.5,Norberto Murara Neto,Bournemouth,10,5,hgb,3.091316,4.3,True,90.0,True,90.000000
11,GK,5,101,4.5,Mark Flekken,Brentford,1,5,hgb,2.619173,1.8,True,90.0,True,90.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
26041,FWD,38,415,8.4,Alexander Isak,Newcastle,14,35,svr,5.918015,7.6,False,80.0,True,77.666667
26047,FWD,38,468,4.7,Chris Wood,Nott'm Forest,13,35,svr,3.229987,5.3,False,78.0,True,89.666667
26054,FWD,38,523,5.7,Michail Antonio,West Ham,2,35,svr,2.046461,2.2,False,80.0,True,77.666667
26061,FWD,38,590,5.5,Matheus Santos Carneiro Da Cunha,Wolves,2,35,svr,2.584901,3.0,False,77.0,True,80.000000


## $\Sigma_\delta$: correlations by position and relationship

Haugh & Singal (Appendix C) split the covariance as $\Sigma_\delta = \mathrm{diag}(\sigma_\delta)\,\rho_\delta\,\mathrm{diag}(\sigma_\delta)$:

* $\sigma_\delta$: each player's standard deviation, updated weekly;
* $\rho_\delta$: correlations **pegged to positions**, not players (RotoViz averages, e.g. a kicker and his own
  team's defence $+0.35$, the opposing defence $-0.50$), fixed and checked to be PSD.

There is no RotoViz table for FPL, so $\rho_\delta$ is estimated from the data. Every pair of players in the same
gameweek is either on the **same team**, **opponents**, or unrelated. For each (relationship, position pair) the
correlation is the average product of the two players' standardised prediction errors
$z = (\text{error} - \bar e)/s_e$, with error $=$ total_points $-$ pred, pooled over every such pair in the
earlier gameweeks. Unrelated pairs get $0$. That is 20 numbers estimated from thousands of pairs each, instead of
one noisy number per player pair from a handful of gameweeks.

$\sigma_\delta$ is as in `mvt.ipynb`: the standard deviation of each player's past errors (at least `MIN_HISTORY`
gameweeks), otherwise his position's median.

In [148]:
MIN_HISTORY = (
    3  # gameweeks of errors a player needs for his own variance / standardisation
)
FIRST_GW = 6  # predictions start at 5
POS_ORDER = ["GK", "DEF", "MID", "FWD"]

SRC = ROOT / "inputs/sources/vaastav_23-24"
team_names = pd.read_csv(SRC / "teams.csv").set_index("id")["name"]
fixtures = pd.read_csv(SRC / "fixtures.csv").dropna(subset=["event"])
fixtures["event"] = fixtures["event"].astype(int)
home, away = fixtures["team_h"].map(team_names), fixtures["team_a"].map(team_names)

# one row per (round, team, opponent): a team with a double gameweek has two rows, a blank none
opponents = pd.concat(
    [
        pd.DataFrame({"round": fixtures["event"], "team": home, "opp": away}),
        pd.DataFrame({"round": fixtures["event"], "team": away, "opp": home}),
    ],
    ignore_index=True,
)
assert set(preds["team"]) <= set(opponents["team"])
opponents.groupby(["round", "team"]).size().value_counts().rename(
    "fixtures per team-round"
)

# opponents

1    714
2     23
Name: fixtures per team-round, dtype: int64

In [149]:
def pos_pair(a, b):
    return tuple(sorted((a, b), key=POS_ORDER.index))


def standardised_errors(hist):
    # each player's errors (total_points - pred), standardised by his own mean and sd
    e = hist[["round", "fpl_id", "pos", "team"]].copy()
    e["error"] = hist["total_points"] - hist["pred"]
    g = e.groupby("fpl_id")["error"]
    e["z"] = (e["error"] - g.transform("mean")) / g.transform("std")
    return e[(g.transform("size") >= MIN_HISTORY) & np.isfinite(e["z"])]


def correlation_table(hist):
    # rho for each (relationship, position pair), pooled over every same-team / opposing pair in `hist`
    e = standardised_errors(hist)
    same = e.merge(e, on=["round", "team"], suffixes=("_i", "_j"))
    same = same[same["fpl_id_i"] < same["fpl_id_j"]].assign(rel="same")
    opp = e.merge(opponents, on=["round", "team"]).merge(
        e, left_on=["round", "opp"], right_on=["round", "team"], suffixes=("_i", "_j")
    )
    opp = opp[opp["fpl_id_i"] < opp["fpl_id_j"]].assign(rel="opp")
    pairs = pd.concat([same, opp], ignore_index=True)
    pairs["pos_pair"] = [pos_pair(a, b) for a, b in zip(pairs["pos_i"], pairs["pos_j"])]
    # print(pairs['pos_pair'])
    return (
        (pairs["z_i"] * pairs["z_j"])
        .groupby([pairs["rel"], pairs["pos_pair"]])
        .agg(rho="mean", n_pairs="size")
    )


season_table = correlation_table(preds)
season_table["rho"].unstack("rel").round(3)

rel,opp,same
pos_pair,,
"(DEF, DEF)",-0.125,0.345
"(DEF, FWD)",-0.167,0.083
"(DEF, MID)",-0.127,0.119
"(FWD, FWD)",-0.020,0.064
"(GK, DEF)",-0.089,0.428
"(GK, FWD)",-0.203,0.015
"(GK, GK)",-0.082,NaN
"(GK, MID)",-0.155,0.130
"(MID, FWD)",-0.053,0.064


Check: pairs with no relationship should be close to 0 if leaving them out is harmless. A large value would mean a
common gameweek effect in the prediction errors.

In [ ]:
e = standardised_errors(preds)
pairs = e.merge(  # .sample(8000, random_state=0).merge(
    e.sample(3000, random_state=1), on="round", suffixes=("_i", "_j")
)
pairs = pairs.merge(
    opponents, left_on=["round", "team_i"], right_on=["round", "team"], how="left"
)

unrelated = pairs[
    (pairs["team_i"] != pairs["team_j"]) & (pairs["opp"] != pairs["team_j"])
]

print(
    f"unrelated pairs: mean z_i z_j = {(unrelated['z_i'] * unrelated['z_j']).mean():.3f} ({len(unrelated):,} pairs)"
)

unrelated pairs: mean z_i z_j = -0.001 (576,008 pairs)


In [ ]:
def nearest_psd_corr(corr, min_eig_frac=1e-4):
    vals, vecs = np.linalg.eigh((corr + corr.T) / 2)
    vals = np.clip(vals, max(min_eig_frac * vals.max(), 1e-8), None)
    c = vecs @ np.diag(vals) @ vecs.T
    d = np.sqrt(np.diag(c))
    return c / np.outer(d, d)


def player_std(players, hist):
    # sd of each player's past errors; position median if fewer than MIN_HISTORY, pooled early on
    errors = hist["total_points"] - hist["pred"]
    var = errors.groupby(hist["fpl_id"]).agg(["var", "size"])
    var = var.loc[var["size"] >= MIN_HISTORY, "var"].reindex(players.index)
    pos_median = var[var > 0].groupby(players["pos"]).median()
    pooled = errors.groupby(hist["pos"]).var()
    pos_median = pos_median.reindex(pooled.index).fillna(pooled)
    return np.sqrt(var.where(var > 0, players["pos"].map(pos_median)).to_numpy())


def build_mu_sigma(gw):
    # mu_delta, Sigma_delta for gameweek gw, using only gameweeks before gw
    players = preds[preds["round"] == gw].drop_duplicates("fpl_id").set_index("fpl_id")
    hist = preds[preds["round"] < gw]
    rho = correlation_table(hist)["rho"]
    std = player_std(players, hist)

    pos, team = players["pos"].to_numpy(), players["team"].to_numpy()
    week = opponents[opponents["round"] == gw]
    plays = set(zip(week["team"], week["opp"]))
    corr = np.eye(len(players))
    for a in range(len(players)):
        for b in range(a + 1, len(players)):
            if team[a] == team[b]:
                rel = "same"
            elif (team[a], team[b]) in plays:
                rel = "opp"
            else:
                continue
            corr[a, b] = corr[b, a] = rho.get((rel, pos_pair(pos[a], pos[b])), 0.0)
    sigma = nearest_psd_corr(corr) * np.outer(std, std)
    return players, players["pred"].to_numpy(), sigma


players, mu_delta, Sigma_delta = build_mu_sigma(20)
print(
    Sigma_delta.shape, "min eigenvalue", np.linalg.eigvalsh(Sigma_delta).min().round(4)
)

(212, 212) min eigenvalue 0.0671


### Out-of-sample check

For each test gameweek, the average Gaussian log-likelihood per player of that week's error vector under three
choices of $\Sigma_\delta$, each built only from earlier gameweeks: the one above, the per-pair sample covariance
`mvt.ipynb` uses now, and the diagonal alone (no correlations). Higher is better.

In [152]:
def sample_sigma(gw, min_eig_frac=1e-4):
    # mvt.ipynb's Sigma: per-pair sample covariance of the errors, repaired to PSD
    players = preds[preds["round"] == gw].drop_duplicates("fpl_id").set_index("fpl_id")
    hist = preds[preds["round"] < gw]
    errors = (hist["total_points"] - hist["pred"]).rename("error")
    pivot = hist.assign(error=errors).pivot_table(
        index="round", columns="fpl_id", values="error"
    )
    cov = pivot.reindex(columns=players.index).cov(min_periods=MIN_HISTORY)
    std = player_std(players, hist)
    corr = np.clip(cov.fillna(0.0).to_numpy() / np.outer(std, std), -1, 1)
    np.fill_diagonal(corr, 1.0)
    return nearest_psd_corr(corr, min_eig_frac) * np.outer(std, std)


def gauss_ll(x, S):
    _, logdet = np.linalg.slogdet(S)
    return -0.5 * (logdet + x @ np.linalg.solve(S, x) + len(x) * np.log(2 * np.pi))


rows = []
for gw in range(FIRST_GW + 3, 39, 3):
    players, mu, S = build_mu_sigma(gw)
    x = (players["total_points"] - players["pred"]).to_numpy()
    rows.append(
        {
            "gw": gw,
            "structured": gauss_ll(x, S) / len(x),
            "sample (mvt)": gauss_ll(x, sample_sigma(gw)) / len(x),
            "diagonal": gauss_ll(x, np.diag(np.diag(S))) / len(x),
        }
    )
ll = pd.DataFrame(rows).set_index("gw")
display(ll.round(2))
ll.mean().round(2)

,structured,sample (mvt),diagonal
gw,,,
9,-3.42,-211.03,-3.36
12,-3.60,-341.73,-3.49
15,-3.76,-425.92,-3.68
18,-3.09,-308.00,-3.29
21,-2.65,-196.84,-2.70
24,-3.34,-416.64,-3.38
27,-2.61,-244.54,-2.65
30,-2.58,-315.84,-2.59
33,-2.70,-385.97,-2.86


structured       -3.04
sample (mvt)   -315.34
diagonal         -3.07
dtype: float64